# HAM10000 ResNet Fine-tuning
This notebook demonstrates how to fine-tune a ResNet model on the HAM10000 dataset using the `ResNetFineTuneTrainer` class.

In [8]:
import importlib
import dualbranch.ResNetFinetuneTrain as rft
importlib.reload(rft)

<module 'dualbranch.ResNetFinetuneTrain' from 'C:\\Users\\T22127A\\Documents\\priv\\priv_dev\\dermascan\\python\\dualbranch\\ResNetFinetuneTrain.py'>

In [9]:
import sys
import os
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

In [10]:
from dualbranch.ham10000_dataset_manager import HAM10000DatasetManager
from dualbranch.ResNetFinetuneTrain import ResNetFineTuneTrainer

In [11]:
# 1. Prepare the Data
# Initialize the dataset manager
data_dir = str(proj_dir / "data")
manager = HAM10000DatasetManager(data_dir=data_dir)

# Load metadata (filtering for Melanoma vs Nevi as per trainer restrictions)
manager.load_cleaned_metadata(filter_melanoma_only=True, sample_size=2226)

# Split the data into train (70%), validation (15%), and test (15%) sets
train_df, val_df, test_df = manager.train_test_val_split(val_size=0.15, test_size=0.15)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

Training samples: 1558
Validation samples: 334
Test samples: 334


In [12]:
# 2. Configure the Trainer
trainer = ResNetFineTuneTrainer()

# Set up the model (ResNet50, 2 classes: nv vs mel)
trainer.set_model(model_name='resnet50', pretrained=True) # , input_channels=6

# Set data loaders
trainer.set_data_loaders(train_df, val_df, test_df, filter_list=['lbp_filter'])

# Configure optimizer and scheduler
trainer.configure_optimizer(optimizer_type='adam', lr=1e-4, weight_decay=1e-4)
trainer.configure_scheduler(scheduler_type='reduce_on_plateau', patience=3)

# Set output paths for artifacts
trainer.set_output_paths(
    model_save_dir="../results/models",
    results_csv_path="../results/training_results.csv",
    history_plot_dir="../results/plots"
)

In [15]:
current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve()

filtered_path = Path(str(proj_dir) + "/data/filtered")
filter_dirs = [p for p in filtered_path.iterdir() if p.is_dir()]

filter_list = ['gabor_filter']

for fdir in filter_dirs:
    if fdir.name in filter_list:
        print(fdir.name)

# for fdir in filter_dirs:
#     print(fdir.name)
#     if fdir.name in self.filter_list:
#         features_file = np.load(filtered_path / fdir.name / (self.dataframe.iloc[idx]['image_id'] + ".npy"))
#         features_tensor = torch.from_numpy(features_file)
#         channel_dim = features_tensor.unsqueeze(0).repeat(3, 1, 1)  # replicate to 3 channels
#         features_map[fdir.name] = channel_dim

#     else:
#         raise ValueError(f"Filter '{fdir.name}' is not in the specified filter list.")

gabor_filter


In [13]:
# 3. Train the Model
# Train for a specified number of epochs
trainer.train(epochs=15)

Epoch 1/15
----------
train Loss: 0.6755 Acc: 0.5866
val Loss: 0.6825 Acc: 0.5180
Epoch 2/15
----------
train Loss: 0.6460 Acc: 0.6746
val Loss: 0.6341 Acc: 0.6737
Epoch 3/15
----------
train Loss: 0.6269 Acc: 0.6990
val Loss: 0.6178 Acc: 0.6946
Epoch 4/15
----------
train Loss: 0.6068 Acc: 0.7105
val Loss: 0.6032 Acc: 0.7006
Epoch 5/15
----------
train Loss: 0.5969 Acc: 0.7137
val Loss: 0.5979 Acc: 0.6796
Epoch 6/15
----------
train Loss: 0.5805 Acc: 0.7240
val Loss: 0.5871 Acc: 0.6946
Epoch 7/15
----------
train Loss: 0.5791 Acc: 0.7137
val Loss: 0.5814 Acc: 0.6976
Epoch 8/15
----------
train Loss: 0.5683 Acc: 0.7266
val Loss: 0.5800 Acc: 0.6946
Epoch 9/15
----------
train Loss: 0.5622 Acc: 0.7368
val Loss: 0.5740 Acc: 0.6826
Epoch 10/15
----------
train Loss: 0.5590 Acc: 0.7246
val Loss: 0.5724 Acc: 0.6886
Epoch 11/15
----------
train Loss: 0.5536 Acc: 0.7330
val Loss: 0.5641 Acc: 0.6856
Epoch 12/15
----------
train Loss: 0.5457 Acc: 0.7356
val Loss: 0.5644 Acc: 0.6916
Epoch 13/15
-

In [14]:
# 4. Evaluate and Save Results
trainer.evaluate()
trainer.save_results_to_csv()

Evaluation results - F1 (macro): 0.7482, F1 (weighted): 0.7482, AUC: 0.8207, Accuracy: 0.7485, Sensitivity: 0.7844, Specificity: 0.7126
